In [0]:
from pyspark.sql import functions as F

# 1. Load the updated Silver table (which now uses MERGE/Upsert)
silver_df = spark.table("nasa_project.silver.asteroids_cleaned")

# 2. Aggregate into high-value metrics
# We perform a full refresh of the Gold table to ensure 'avg' and 'min' 
# metrics are accurate across the entire history.
gold_df = silver_df.groupBy("date").agg(
    F.count("name").alias("total_asteroids"),
    F.sum(F.col("is_hazardous").cast("int")).alias("hazardous_count"),
    F.avg("miss_km").alias("avg_miss_km"),
    F.min("miss_km").alias("closest_approach_km"),
    F.max("size_km").alias("max_asteroid_size_km")
).orderBy(F.col("date").desc())

# 3. Save to Gold
# Since this is a small summary table, 'overwrite' with 'overwriteSchema' 
# is the safest way to keep the dashboard fresh daily.
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nasa_project.gold.daily_summary")

# 4. Display results for your portfolio screenshot
print(f"Gold Layer refreshed. Total dates tracked: {gold_df.count()}")
display(gold_df)